# PAV_Analysis

Pavlovian analysis: merged session CSVs → per-trial metrics, event-locked traces and learning contrasts, saved in one pickle.

**Pipeline:** 1. PAV_Preprocessing → **2. PAV_Analysis** → 3. PAV_Publication_Figure

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/Pavlovian/RWD/Processed Data/` | 48 merged CSVs (24 mice × early session `01` and late session `02`) |

### Outputs (downloaded at the end)
`pav_pub_data.pkl`, for `Photometry/Pavlovian/`, containing:
- `pav_df`: per mouse × session metrics (tones, grabs, % collected, latency, cue and grab peak / mean / AUC dopamine, speed, distance)
- `psth_animals`, `trial_evt`, `trial_psths`: per-mouse and per-trial traces around tone onset and pellet grab
- `delta_df`: late − early change per mouse
- `DA_WINDOWS` and display settings

### How to run
Run all cells top to bottom (a few minutes). `SKIP_EXISTING=True` stops if the pickle is already committed; set it to `False` to recompute. The run is pinned to the current commit of `main`, which is printed.

### Measurement windows
| Event | Window | |
|---|---|---|
| Tone (cue) | 0 to +5 s | the whole 5 s tone |
| Pellet grab | −2 to +1 s | the grab response peaks just before the grab |

A tone counts as collected if a grab follows within 30 s of tone onset.

In [ ]:
# @title Setup — imports
import io
import os
import re
import pickle
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm

print("Imports OK.")


Imports OK.


In [ ]:
# @title Output path & run options
GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr" if you point at a personal fork
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_API = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
PAV    = "Photometry/Pavlovian"

# ── Pin this run to one commit ────────────────────────────────────────────────
# raw.githubusercontent caches BRANCH urls (~5 min) and edge nodes expire
# independently, so fetching ".../main/..." right after a push can return a MIX of
# old and new files inside one run. Resolve main to a SHA and fetch by SHA: those
# urls are immutable, so a cache hit is always correct. The printed SHA is also the
# provenance record for which data produced this pickle.
DATA_SHA = requests.get(f"{GH_API}/commits/main").json()['sha']
print(f"Data version (main @ commit): {DATA_SHA[:10]}")

# NB: the committed folder name contains a space.
GITHUB_PATH    = f"{PAV}/RWD/Processed%20Data"
GITHUB_API_URL = f"{GH_API}/contents/{GITHUB_PATH}?ref={DATA_SHA}"

PKL_OUT = '/content/pav_pub_data.pkl'   # Colab-local; downloaded at the end

# Skip the whole (slow) analysis if the pickle is already committed to the repo.
# Asked via the API, not the raw CDN, so the answer is never a stale cache hit.
SKIP_EXISTING = False
if SKIP_EXISTING:
    _r = requests.get(f"{GH_API}/contents/{PAV}/pav_pub_data.pkl?ref={DATA_SHA}")
    if _r.status_code == 200:
        raise SystemExit(
            f"[SKIP] pav_pub_data.pkl is already committed at {DATA_SHA[:10]}.\n"
            f"        Set SKIP_EXISTING=False to recompute it.")
print(f"Pickle will be written to: {PKL_OUT}")


In [ ]:
# @title Parameters & configuration

GROUPS   = ['SHAM', 'SNI', 'SK3']
SESSIONS = ['01', '02']          # 01 = early training, 02 = late
GROUP_COLORS = {'SHAM': '#4C72B0', 'SNI': '#DD8452', 'SK3': '#55A868'}
PRES_FS = {'suptitle': 16, 'title': 13, 'label': 12, 'tick': 11,
           'legend': 10, 'annot': 9, 'sig': 10}

TONE_DUR_S = 5.0                 # every Pavlovian session used 5 s tones
DA_COL     = 'Corrected_Fluorescence_Z_25'

# ── Event-locked PSTH window (tight, +/-5 s around each event) ────────────────
EVT_WIN_START, EVT_WIN_END, EVT_POINTS = -5, 5, 200
event_time = np.linspace(EVT_WIN_START, EVT_WIN_END, EVT_POINTS)

# ── Per-trial metric windows ─────────────────────────────────────────────────
# ── DA measurement windows, per event, in seconds relative to the event ──────
# Matching the Variable Tone convention. The GRAB window starts BEFORE the event:
# the grab response peaks at -0.08 to -0.73 s across every group and session, so a
# forward-only window would clip the peak entirely. The tone window spans the 5 s
# tone exactly.
#   NB the two windows differ in length (5 s vs 3 s), so AUC is NOT comparable
#   BETWEEN events -- peak and mean are. All planned comparisons are between-subject
#   within an event, so this does not affect them.
DA_WINDOWS = {'cue': (0.0, 5.0), 'grab': (-2.0, 1.0)}
DIST_TAIL  = 2.0    # s before tone end, for the DistFromFED average
MAX_WAIT   = 30.0   # s after tone onset to look for the pellet grab

# ── Heatmap traces (must match what the figure expects) ──────────────────────
PRE_HM, WIN_END, N_PTS = 1.0, 24.0, 300
hm_time = np.linspace(-PRE_HM, WIN_END, N_PTS)

_NAME_RE = re.compile(r'([MF]\d+)_(SHAM|SNI|SK3)_(C57|DAT)_(\d{6})_(\d{2})')

def parse_name(fname):
    """-> dict(animal, group, genotype, date, session) or None."""
    m = _NAME_RE.search(fname)
    if not m: return None
    return {'animal': m.group(1), 'group': m.group(2), 'genotype': m.group(3),
            'date': m.group(4), 'session': m.group(5)}

print(f"Config ready. groups={GROUPS} sessions={SESSIONS}")


Config ready. groups=['SHAM', 'SNI', 'SK3'] sessions=['01', '02']


In [ ]:
# @title Load the merged CSVs from GitHub

def load_file(file_info):
    r = requests.get(file_info['download_url']); r.raise_for_status()
    meta = parse_name(file_info['name']) or {}
    return {'filename': file_info['name'], 'meta': meta,
            'df': pd.read_csv(io.StringIO(r.text), low_memory=False)}

results = []
resp = requests.get(GITHUB_API_URL); resp.raise_for_status()
csv_files = [f for f in resp.json() if f['name'].endswith('.csv')]
with ThreadPoolExecutor(max_workers=10) as ex:
    futs = {ex.submit(load_file, f): f['name'] for f in csv_files}
    with tqdm(total=len(csv_files), desc="Loading CSVs (GitHub)") as pbar:
        for fut in as_completed(futs):
            try: results.append(fut.result())
            except Exception as e: print(f"  x {futs[fut]}: {e}")
            pbar.update(1)

pav_all = sorted([r for r in results if r['meta']], key=lambda x: x['filename'])
print(f"Loaded {len(pav_all)} Pavlovian sessions from GitHub @ {DATA_SHA[:10]}.")

_by_sess = pd.Series([e['meta']['session'] for e in pav_all]).value_counts().to_dict()
print(f"  by session: {_by_sess}")
if len(_by_sess) < 2:
    print("\n!! Only one session is present, so there is no learning comparison to make.\n"
          "   Run PAV_Preprocessing with SKIP_EXISTING=False and commit all 48 merged\n"
          "   CSVs before running this notebook.")


Loading CSVs (GitHub):   0%|          | 0/49 [00:00<?, ?it/s]

Loaded 48 Pavlovian sessions from GitHub @ db96b2607a.
  by session: {'01': 24, '02': 24}


In [ ]:
# @title Helper functions

def event_times(df, col):
    """Timestamps where a binary event column is 1."""
    if col not in df.columns: return np.array([])
    v = pd.to_numeric(df[col], errors='coerce').values
    return df['TimeStampAdjusted'].values[v == 1]

def make_interp(df, col):
    """Interpolator over a signal column, NaN outside the recording."""
    if col not in df.columns: return None
    ts = df['TimeStampAdjusted'].values
    y  = pd.to_numeric(df[col], errors='coerce').values
    ok = np.isfinite(ts) & np.isfinite(y)
    if ok.sum() < 10: return None
    return interp1d(ts[ok], y[ok], kind='linear', bounds_error=False, fill_value=np.nan)

def grab_after(tone_t, grab_times, max_wait=None):
    """First pellet grab after a tone onset, or nan if the animal never collected."""
    mw = MAX_WAIT if max_wait is None else max_wait
    later = grab_times[(grab_times >= tone_t) & (grab_times <= tone_t + mw)]
    return float(later[0]) if len(later) else np.nan

def window_stat(fi, t0, ev):
    """(peak, mean, auc) of the DA trace over this event's own window.

    AUC is the trapezoid integral in z-score x seconds. Within one event it is just
    mean x window length, so the two are not independent evidence."""
    if fi is None or not np.isfinite(t0):
        return np.nan, np.nan, np.nan
    a, b = DA_WINDOWS[ev]
    tt = np.linspace(t0 + a, t0 + b, 120)
    v  = fi(tt)
    ok = np.isfinite(v)
    if ok.sum() < 5:
        return np.nan, np.nan, np.nan
    return (float(np.nanmax(v[ok])), float(np.nanmean(v[ok])),
            float(np.trapezoid(v[ok], tt[ok])))

print("Helpers defined.")


Helpers defined.


In [ ]:
# @title Per-trial metrics -> pav_df (per animal x session)

records = []
for entry in tqdm(pav_all, desc="Per-trial metrics"):
    df, m = entry['df'], entry['meta']
    if DA_COL not in df.columns or 'TimeStampAdjusted' not in df.columns:
        continue
    fi_da    = make_interp(df, DA_COL)
    fi_speed = make_interp(df, 'Speed_cm')
    fi_dist  = make_interp(df, 'DistFromFED')
    tones = event_times(df, 'ToneStart')
    grabs = event_times(df, 'PelletGrab')

    trials = []
    for t0 in tones:
        tg      = grab_after(t0, grabs)
        cue_pk, cue_mn, cue_auc = window_stat(fi_da, t0, 'cue')
        gr_pk,  gr_mn,  gr_auc  = window_stat(fi_da, tg, 'grab')
        # speed during the tone; distance from FED over the tail of the tone
        spd = np.nan
        if fi_speed is not None:
            v = fi_speed(np.linspace(t0, t0 + TONE_DUR_S, 40))
            if np.isfinite(v).any(): spd = float(np.nanmean(v))
        dist = np.nan
        if fi_dist is not None:
            v = fi_dist(np.linspace(t0 + TONE_DUR_S - DIST_TAIL, t0 + TONE_DUR_S, 30))
            if np.isfinite(v).any(): dist = float(np.nanmean(v))
        trials.append({'ret_latency_s': tg - t0 if np.isfinite(tg) else np.nan,
                       'cue_peak_da': cue_pk, 'cue_mean_da': cue_mn, 'cue_auc': cue_auc,
                       'grab_peak_da': gr_pk, 'grab_mean_da': gr_mn, 'grab_auc': gr_auc,
                       'mean_speed_cm_s': spd, 'dist_at_end_cm': dist})

    if not trials: continue
    td = pd.DataFrame(trials)
    rec = {'animal': m['animal'], 'group': m['group'], 'session': m['session'],
           'n_tones': int(len(tones)), 'n_grabs': int(len(grabs)),
           # fraction of tones the animal actually collected -- the behavioural
           # learning measure, independent of the photometry
           'pct_collected': round(100 * td['ret_latency_s'].notna().mean(), 1)}
    for c in td.columns:
        rec[c] = float(td[c].mean()) if td[c].notna().any() else np.nan
    records.append(rec)

pav_df = pd.DataFrame(records).sort_values(['group','animal','session']).reset_index(drop=True)
print(f"pav_df: {len(pav_df)} rows (animal x session)")
print(pav_df.groupby(['group','session']).size().to_string())
pav_df.head()


Per-trial metrics:   0%|          | 0/48 [00:00<?, ?it/s]

pav_df: 48 rows (animal x session)
group  session
SHAM   01         8
       02         8
SK3    01         8
       02         8
SNI    01         8
       02         8


,animal,group,session,n_tones,n_grabs,pct_collected,ret_latency_s,cue_peak_da,cue_mean_da,cue_auc,grab_peak_da,grab_mean_da,grab_auc,mean_speed_cm_s,dist_at_end_cm
0,F18,SHAM,01,10,8,40.0,22.280457,9.973486,2.513373,12.553234,34.617744,16.721824,50.438526,8.467292,7.584923
1,F18,SHAM,02,18,15,77.8,10.727192,15.073199,5.363931,26.888070,27.317977,10.028172,30.167644,7.641638,6.122913
2,F19,SHAM,01,11,6,9.1,7.932281,25.496385,8.568498,43.114910,24.408303,14.156966,42.748804,6.303621,13.765211
3,F19,SHAM,02,15,10,66.7,7.798997,12.578760,5.185114,26.047020,18.303110,9.462227,28.464997,5.937683,9.883011
4,F2,SHAM,01,6,6,16.7,29.596121,2.866435,-0.420287,-2.150099,4.743188,2.205522,6.607220,4.634091,6.369973


In [ ]:
# @title Event-locked PSTHs -> psth_animals[event][group][session]

psth_animals = {ev: {g: {s: [] for s in SESSIONS} for g in GROUPS}
                for ev in ['Cue', 'PelletGrab']}

for entry in tqdm(pav_all, desc="PSTHs"):
    df, m = entry['df'], entry['meta']
    g, s = m['group'], m['session']
    if g not in GROUPS or s not in SESSIONS: continue
    fi = make_interp(df, DA_COL)
    if fi is None: continue
    for ev, col in (('Cue', 'ToneStart'), ('PelletGrab', 'PelletGrab')):
        et = event_times(df, col)
        if not len(et): continue
        seg = np.array([fi(t + event_time) for t in et])
        if seg.size and np.isfinite(seg).any():
            psth_animals[ev][g][s].append(np.nanmean(seg, axis=0))

for ev in psth_animals:
    print(f"  {ev:11s} " + "  ".join(
        f"{g} s{s}: {len(psth_animals[ev][g][s])}" for g in GROUPS for s in SESSIONS))


PSTHs:   0%|          | 0/48 [00:00<?, ?it/s]

  Cue         SHAM s01: 8  SHAM s02: 8  SNI s01: 8  SNI s02: 8  SK3 s01: 8  SK3 s02: 8
  PelletGrab  SHAM s01: 8  SHAM s02: 8  SNI s01: 8  SNI s02: 8  SK3 s01: 8  SK3 s02: 8


In [ ]:
# @title Per-trial traces for the figure (heatmaps + distance)
# Precomputed here so the figure notebook needs only the pickle, never the CSVs.
# NOTE: the window is baked in -- changing PRE_HM / WIN_END / N_PTS means re-running
# this notebook, not just the figure.

trial_psths      = {s: {g: [] for g in GROUPS} for s in SESSIONS}
trial_dist_tone  = {s: {g: [] for g in GROUPS} for s in SESSIONS}
animal_dist_tone = {s: {} for s in SESSIONS}
# Per-trial traces locked to EACH event on the tight event_time grid, for the
# figure's tone-onset and pellet-retrieval heatmaps.
trial_evt = {ev: {s: {g: [] for g in GROUPS} for s in SESSIONS}
             for ev in ['Cue', 'PelletGrab']}

for entry in tqdm(pav_all, desc="Per-trial traces"):
    df, m = entry['df'], entry['meta']
    g, s, aid = m['group'], m['session'], m['animal']
    if g not in GROUPS or s not in SESSIONS: continue
    if DA_COL not in df.columns: continue

    ts = pd.to_numeric(df['TimeStampAdjusted'], errors='coerce').values
    z  = pd.to_numeric(df[DA_COL], errors='coerce').values
    has_dist = 'DistFromFED' in df.columns
    dist = pd.to_numeric(df['DistFromFED'], errors='coerce').values if has_dist else None

    tones = event_times(df, 'ToneStart')
    for t0 in tones:
        mask  = (ts >= t0 - PRE_HM - 0.5) & (ts <= t0 + WIN_END + 0.5)
        t_rel, z_win = ts[mask] - t0, z[mask]
        ok = np.isfinite(z_win)
        if ok.sum() < 10: continue
        trial_psths[s][g].append((aid, np.interp(hm_time, t_rel[ok], z_win[ok])))

    for ev, col in (('Cue', 'ToneStart'), ('PelletGrab', 'PelletGrab')):
        for t0 in event_times(df, col):
            mask = (ts >= t0 + EVT_WIN_START - 0.5) & (ts <= t0 + EVT_WIN_END + 0.5)
            t_rel, z_win = ts[mask] - t0, z[mask]
            ok = np.isfinite(z_win)
            if ok.sum() < 10: continue
            trial_evt[ev][s][g].append((aid, np.interp(event_time, t_rel[ok], z_win[ok])))

    if has_dist and len(tones):
        vals = []
        for t0 in tones:
            v = dist[np.argmin(np.abs(ts - (t0 + TONE_DUR_S)))]
            if np.isfinite(v): vals.append(float(v))
        if vals:
            trial_dist_tone[s][g].extend(vals)
            animal_dist_tone[s][aid] = {'group': g, 'mean': float(np.mean(vals))}

print("Per-trial heatmap rows:",
      {s: {g: len(trial_psths[s][g]) for g in GROUPS} for s in SESSIONS})
print("Animals with distance:", {s: len(animal_dist_tone[s]) for s in SESSIONS})
for ev in trial_evt:
    print(f"Per-trial {ev:10s} rows:",
          {s: {g: len(trial_evt[ev][s][g]) for g in GROUPS} for s in SESSIONS})


Per-trial traces:   0%|          | 0/48 [00:00<?, ?it/s]

Per-trial heatmap rows: {'01': {'SHAM': 62, 'SNI': 72, 'SK3': 46}, '02': {'SHAM': 173, 'SNI': 159, 'SK3': 141}}
Animals with distance: {'01': 24, '02': 20}
Per-trial Cue        rows: {'01': {'SHAM': 62, 'SNI': 72, 'SK3': 46}, '02': {'SHAM': 173, 'SNI': 159, 'SK3': 141}}
Per-trial PelletGrab rows: {'01': {'SHAM': 40, 'SNI': 50, 'SK3': 37}, '02': {'SHAM': 155, 'SNI': 141, 'SK3': 118}}


In [ ]:
# @title Learning contrasts — delta peak z and paired AUC (Trained - Baseline)
# Session 01 = baseline, 02 = trained. These are the per-mouse learning effects the
# figure annotates: how much each animal's response CHANGED with training, paired
# within animal so between-animal variability drops out.

_M = ['cue_peak_da','cue_mean_da','cue_auc','grab_peak_da','grab_mean_da','grab_auc']

_w = pav_df.pivot_table(index=['animal','group'], columns='session', values=_M)
rows = []
for (aid, grp), r in _w.iterrows():
    rec = {'animal': aid, 'group': grp}
    for m in _M:
        bl = r.get((m, '01'), np.nan)
        tr = r.get((m, '02'), np.nan)
        rec[f'{m}_baseline'] = bl
        rec[f'{m}_trained']  = tr
        rec[f'{m}_delta']    = tr - bl          # Trained - Baseline
    rows.append(rec)
delta_df = pd.DataFrame(rows).sort_values(['group','animal']).reset_index(drop=True)

_paired = delta_df[[f'{m}_delta' for m in _M]].notna().all(axis=1)
print(f"delta_df: {len(delta_df)} animals, {int(_paired.sum())} with BOTH sessions on every metric")
print(delta_df.groupby('group').size().rename('animals').to_string())
print()

print("=== delta (Trained - Baseline), mean +/- SEM per group ===")
print(f"  {'metric':16s} " + "  ".join(f"{g:>16s}" for g in GROUPS))
for m in _M:
    cells = []
    for g in GROUPS:
        v = delta_df.loc[delta_df.group == g, f'{m}_delta'].dropna().values
        cells.append("        n/a     " if len(v) < 2 else
                     f"{v.mean():+7.2f}+/-{v.std(ddof=1)/np.sqrt(len(v)):.2f}")
    print(f"  {m:16s} " + "  ".join(f"{c:>16s}" for c in cells))

# ── stats: is the CHANGE different between groups, and non-zero within one? ───
try:
    from scipy.stats import f_oneway, ttest_rel, ttest_1samp
    print()
    print("=== one-way ANOVA across groups on the delta ===")
    for m in _M:
        gs = [delta_df.loc[delta_df.group == g, f'{m}_delta'].dropna().values for g in GROUPS]
        gs = [x for x in gs if len(x) > 1]
        if len(gs) > 1:
            F, pv = f_oneway(*gs)
            print(f"  {m:16s} F={F:6.3f}  p={pv:.4f}{'  *' if pv < 0.05 else ''}")
    print()
    print("=== within group: is Trained different from Baseline? (paired t) ===")
    for g in GROUPS:
        for m in _M:
            d = delta_df[delta_df.group == g]
            a = d[f'{m}_baseline'].values; b = d[f'{m}_trained'].values
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() > 2:
                t, pv = ttest_rel(a[ok], b[ok])
                if pv < 0.05:
                    print(f"  {g:5s} {m:16s} n={int(ok.sum())}  "
                          f"{a[ok].mean():6.2f} -> {b[ok].mean():6.2f}  p={pv:.4f}  *")
except ImportError:
    print("scipy unavailable -- skipping stats")


delta_df: 24 animals, 9 with BOTH sessions on every metric
group
SHAM    8
SK3     8
SNI     8

=== delta (Trained - Baseline), mean +/- SEM per group ===
  metric                       SHAM               SNI               SK3
  cue_peak_da          +3.96+/-3.76      +4.02+/-3.88      +7.41+/-5.05
  cue_mean_da          +4.55+/-1.97      +4.57+/-1.81      +5.48+/-2.06
  cue_auc             +22.84+/-9.88     +22.93+/-9.09    +27.51+/-10.37
  grab_peak_da         -3.26+/-4.13      -8.65+/-9.05      -4.69+/-2.01
  grab_mean_da         -3.43+/-3.25      -5.05+/-4.85      -1.78+/-0.40
  grab_auc            -10.36+/-9.77    -15.19+/-14.60      -5.41+/-1.22

=== one-way ANOVA across groups on the delta ===
  cue_peak_da      F= 0.214  p=0.8090
  cue_mean_da      F= 0.075  p=0.9280
  cue_auc          F= 0.074  p=0.9287
  grab_peak_da     F= 0.218  p=0.8103
  grab_mean_da     F= 0.146  p=0.8673
  grab_auc         F= 0.144  p=0.8686

=== within group: is Trained different from Baseline? (paired 

In [ ]:
# @title Package outputs, save the pickle & download it

_save = {
    'pav_df':           pav_df,             # per animal x session metrics
    'event_time':       event_time,
    'psth_animals':     psth_animals,       # {event: {group: {session: [traces]}}}
    'trial_psths':      trial_psths,        # {session: {group: [(animal, trace)]}}
    'trial_evt':        trial_evt,          # {event: {session: {group: [(animal, trace)]}}} on event_time
    'trial_dist_tone':  trial_dist_tone,    # {session: {group: [float]}}
    'animal_dist_tone': animal_dist_tone,   # {session: {animal: {group, mean}}}
    'hm_time':          hm_time,
    'delta_df':         delta_df,           # per-animal Trained - Baseline contrasts
    'DA_WINDOWS':       DA_WINDOWS,         # the per-event measurement windows
    'GROUPS':           GROUPS,
    'SESSIONS':         SESSIONS,
    'GROUP_COLORS':     GROUP_COLORS,
    'PRES_FS':          PRES_FS,
}

with open(PKL_OUT, 'wb') as f:
    pickle.dump(_save, f)

print(f"[SAVED] {PKL_OUT}  ({os.path.getsize(PKL_OUT)/1e6:.2f} MB)")
print("Keys:", list(_save.keys()))

# Colab-local output does not survive the runtime -- pull it down, then commit it to
# Photometry/Pavlovian/pav_pub_data.pkl yourself.
from google.colab import files
files.download(PKL_OUT)
